# Destiny Studio — production autonome Colab Pro

Un seul notebook : Atlas → Blender → narration **française Kokoro** → sous-titres réellement alignés → Remotion → MP4 H.264/AAC dans Drive. **Aucun job GitHub Actions n’est requis.** Les modèles Oryx/Rhulk/Nezarec/Cropta restent des prototypes ; Athéon/Templier sont préservés.

Sur Android Chrome : ouvrez Colab, connectez-vous, choisissez **Exécution → Modifier le type d’exécution → GPU T4** si disponible. Exécutez les cellules dans l’ordre. Un GPU demandé n’est pas une validation GPU : le backend effectif figure dans les rapports. Le repli CPU est explicite et peut être lent.

Première installation : téléchargements réseau des outils et modèles vocaux. Aucun PC local, clé API ou terminal n’est nécessaire. Ne lancez pas une longue vidéo avant le smoke-test de 2 secondes.

In [ ]:
# @title 1. Monter Google Drive et installer le studio autonome
from google.colab import drive, output
from pathlib import Path
import hashlib, json, os, subprocess, sys, tarfile, urllib.request
from IPython.display import display, Video, FileLink
output.enable_custom_widget_manager()
drive.mount('/content/drive')
projet_a_reprendre = '' # @param {type:"string"}
DRIVE_ROOT = Path('/content/drive/MyDrive')
REPO = Path('/content/Destiny-Explained-Studio')
BRANCH = 'feat/standalone-colab-production-studio'
def run(args, **kwargs):
    subprocess.run([str(x) for x in args], check=True, **kwargs)
if not REPO.exists():
    run(['git','clone','--depth','1','--single-branch','--branch',BRANCH,'https://github.com/Alexanst54/Destiny-Explained-Studio.git',REPO])
else:
    actual = subprocess.check_output(['git','-C',REPO,'branch','--show-current'],text=True).strip()
    assert actual in [BRANCH,''], 'Dossier sur une autre branche : utilisez un runtime neuf.'
if projet_a_reprendre:
    import re
    project=Path(projet_a_reprendre).expanduser().resolve()
    assert project.is_relative_to(DRIVE_ROOT.resolve()), 'Projet attendu dans Drive.'
    manifest=project/'project.json'
    if not manifest.exists(): manifest=project/'smoke/project.json'
    saved=json.loads(manifest.read_text());revision=saved.get('sourceCommit','')
    assert re.fullmatch(r'[0-9a-f]{40}',revision) and not saved.get('sourceDirty',True), 'Source de projet non épinglée : utilisez un nouveau projet.'
    run(['git','-C',REPO,'fetch','--depth','1','origin',revision])
    run(['git','-C',REPO,'checkout','--detach',revision])
print('Source épinglée dans ce runtime :',subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip())
run(['apt-get','update','-qq'])
asound='libasound2t64' if subprocess.run(['apt-cache','show','libasound2t64'],capture_output=True).returncode==0 else 'libasound2'
run(['apt-get','install','-y','-qq','ffmpeg','espeak-ng','python3-venv','libxi6','libxrender1','libxkbcommon0','libsm6','libice6','libgl1','libegl1','libnss3','libgbm1','fonts-dejavu-core',asound])
def install_archive(url, checksum_url, name, destination):
    archive=Path('/content')/name
    urllib.request.urlretrieve(url,str(archive))
    checks=urllib.request.urlopen(checksum_url).read().decode()
    digest=hashlib.file_digest(archive.open('rb'),'sha256').hexdigest()
    assert any(digest in line and name in line for line in checks.splitlines()), 'Téléchargement invalide : SHA-256 incorrect.'
    # Official, SHA-verified archives only.
    run(['tar','-xf',archive,'-C',destination])
    archive.unlink()
NODE_DIR=Path('/content/node-v22.23.4-linux-x64')
if not NODE_DIR.exists():
    base='https://nodejs.org/dist/v22.23.4/'
    install_archive(base+'node-v22.23.4-linux-x64.tar.xz',base+'SHASUMS256.txt','node-v22.23.4-linux-x64.tar.xz','/content')
os.environ['PATH']=str(NODE_DIR/'bin')+':'+os.environ['PATH']
os.environ['HF_HOME']='/content/destiny-model-cache/huggingface'
os.environ['DESTINY_MODEL_CACHE']='/content/destiny-model-cache'
NODE=NODE_DIR/'bin/node'
BLENDER=Path('/content/blender-4.3.2-linux-x64/blender')
if not BLENDER.exists():
    base='https://download.blender.org/release/Blender4.3/'
    install_archive(base+'blender-4.3.2-linux-x64.tar.xz',base+'blender-4.3.2.sha256','blender-4.3.2-linux-x64.tar.xz','/content')
VENV=Path('/content/destiny-studio-venv')
if not VENV.exists(): run([sys.executable,'-m','venv',VENV])
PYTHON=VENV/'bin/python'
run([PYTHON,'-m','pip','install','--no-cache-dir','--index-url','https://download.pytorch.org/whl/cpu','torch==2.6.0','torchaudio==2.6.0'])
run([PYTHON,'-m','pip','install','--no-cache-dir','-r',REPO/'remotion/studio/requirements.txt'])
run(['npm','ci'],cwd=REPO/'remotion')
run([NODE,REPO/'remotion/node_modules/@remotion/cli/remotion-cli.js','browser','ensure'],cwd=REPO/'remotion')
try: run(['nvidia-smi'])
except (FileNotFoundError,subprocess.CalledProcessError): print('ATTENTION : GPU NVIDIA indisponible. Blender utilisera le CPU si nécessaire.')
sys.path.insert(0,str(REPO/'remotion/studio'))
from colab_ui import form
print('Installation terminée. Aucun appel à GitHub Actions.')


In [ ]:
# @title 2. Choisir le boss et enregistrer le projet dans Drive
# Les listes d’arène, d’animation et de caméra changent avec le boss.
STUDIO = form(DRIVE_ROOT)


Cliquez **Enregistrer le projet** après vos choix. Le dossier contient `selection.json`, puis un manifeste immuable, la voix, les frames, les journaux et le MP4.

Pour reprendre un projet existant sans ressaisir le formulaire, utilisez la cellule suivante. Changer la narration, durée, qualité, sélection ou source exige un nouveau nom de projet ; on ne réutilise jamais silencieusement des frames incompatibles.

In [ ]:
# @title 2b. Reprendre un projet Drive existant (facultatif)
reprendre = False # @param {type:"boolean"}
dossier = '/content/drive/MyDrive/Destiny-Studio/destiny-001' # @param {type:"string"}
if reprendre:
    saved=Path(dossier).expanduser().resolve()
    assert saved.is_relative_to(DRIVE_ROOT.resolve()), 'Le projet doit être dans votre Drive.'
    assert (saved/'selection.json').exists(), 'selection.json introuvable.'
    STUDIO={'folder':str(saved),'config':str(saved/'selection.json')}
    print('Projet restauré :',saved)


In [ ]:
# @title 3. Smoke-test obligatoire — 2 secondes, narration française et MP4
assert STUDIO.get('config'), 'Enregistrez ou restaurez un projet avant cette étape.'
COMMAND=[PYTHON,REPO/'remotion/studio/production.py',STUDIO['config'],STUDIO['folder'],'--blender',BLENDER,'--node',NODE]
try:
    run(COMMAND+['--smoke'])
    SMOKE=Path(STUDIO['folder'])/'smoke'
    display(Video(str(SMOKE/'video.mp4'),embed=True))
    print(json.loads((SMOKE/'validation.json').read_text()))
except Exception as error:
    print('Smoke-test interrompu. Corrigez la cause puis relancez cette même cellule. Les caches valides sont conservés. Journaux :',Path(STUDIO['folder'])/'smoke/logs')
    raise


**Inspectez le smoke-test avant de poursuivre** : bonne identité, cadrage, mouvement, parole française audible et sous-titres synchronisés. Le smoke utilise la phrase courte « Voici ce boss. », le même boss/animation/caméra, Draft et 320×240. Il ne valide pas la qualité finale ou le matériel T4 à lui seul.

La narration complète sera synthétisée et alignée avant les images de production. Si elle dépasse la durée, aucune image de production n’est rendue : choisissez un nouveau projet avec une durée adaptée. Les effets et matériaux ne perdent pas de géométrie en basse résolution : LOD0 reste explicite.

In [ ]:
# @title 4. Production complète ou reprise — après inspection du smoke-test
smoke_inspecte = False # @param {type:"boolean"}
assert smoke_inspecte, 'Visionnez le smoke-test et cochez smoke_inspecte.'
assert STUDIO.get('config'), 'Enregistrez ou restaurez un projet.'
try:
    run([PYTHON,REPO/'remotion/studio/production.py',STUDIO['config'],STUDIO['folder'],'--blender',BLENDER,'--node',NODE])
except Exception:
    print('Production interrompue. Consultez les logs dans Drive puis relancez cette cellule sans modifier le projet. Les PNG complets sont vérifiés avant réemploi ; une frame corrompue est recalculée.')
    raise


In [ ]:
# @title 5. Vérifier, visionner et télécharger le MP4 final
from google.colab import files
FOLDER=Path(STUDIO['folder'])
report=json.loads((FOLDER/'validation.json').read_text())
assert report.get('status')=='passed', 'La validation finale ne passe pas : consultez les logs.'
# Revalidate the actual file before displaying or downloading it.
from production import validate, verify, sha
config=validate(json.loads((FOLDER/'selection.json').read_text()))
verify(FOLDER/'video.mp4',config)
assert sha(FOLDER/'video.mp4')==report['sha256'], 'MP4 modifié ou incomplet.'
print(json.dumps(report,ensure_ascii=False,indent=2))
display(Video(str(FOLDER/'video.mp4'),embed=True))
telecharger = False # @param {type:"boolean"}
if telecharger: files.download(str(FOLDER/'video.mp4'))
print('Copie permanente déjà enregistrée dans Drive :',FOLDER/'video.mp4')


## Récupération et acceptation Colab / T4

- Session interrompue : renseignez le chemin du projet dans `projet_a_reprendre` de la cellule 1 ; elle restaure le commit enregistré. Reconnectez Drive, puis utilisez la restauration 2b et relancez la même étape. Gardez le même commit, projet et Blender. Aucun cache ne dépend de GitHub Actions. Une modification de source est refusée : épinglez le commit de `project.json` ou utilisez un nouveau dossier.
- GPU : vérifiez les noms des appareils et `backend` OPTIX/CUDA dans `validation.json` et les logs. Le rapport distingue chaque frame réellement rendue sur CPU/GPU, y compris après repli. L’affichage `nvidia-smi` seul n’est pas une preuve de rendu GPU.
- Reprise : relancez la production terminée et vérifiez les `cacheHits` de chaque `blender/chunk-*/progress.json`. Les PNG valides ne sont pas recalculés. Après corruption volontaire d’une **copie de test**, une seule frame doit être recalculée.
- Sur T4 réelle : smoke-test, inspection audio/visuelle Android, clip court Balanced, interruption/reprise, vérification du MP4 et du backend effectif. Archivez les rapports et votre résultat d’inspection dans Drive. Un test CPU local n’est pas une validation Colab/T4/Android.
- Musique : facultative, uniquement un fichier que vous fournissez avec sa provenance/licence ; aucune musique du jeu n’est téléchargée.
- Pas assez d’espace / erreur réseau / mémoire GPU : corrigez la cause, choisissez un nouveau projet plus léger si les réglages changent. Une erreur de modèle ou d’alignement ne déclenche aucun faux modèle ni sous-titre inventé.

Guide complet et limites : `docs/production-studio/README.md` dans le dépôt. Les anciens épisodes, modèles et narrations restent inchangés.